In [1]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

root
 |-- _c0: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- TP2: double (nullable = true)
 |-- TP3: double (nullable = true)
 |-- H1: double (nullable = true)
 |-- DV_pressure: double (nullable = true)
 |-- Reservoirs: double (nullable = true)
 |-- Oil_temperature: double (nullable = true)
 |-- Motor_current: double (nullable = true)
 |-- COMP: integer (nullable = true)
 |-- DV_electric: integer (nullable = true)
 |-- Towers: integer (nullable = true)
 |-- MPG: integer (nullable = true)
 |-- LPS: integer (nullable = true)
 |-- Pressure_switch: integer (nullable = true)
 |-- Oil_level: integer (nullable = true)
 |-- Caudal_impulses: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- dow: integer (nullable = true)

Tổng số bản ghi: 1516948


In [2]:
# PHÂN TÍCH KHUNG GIỜ CAO ĐIỂM VÀ THẤP ĐIỂM CỦA HỆ THỐNG
q2_combined = spark.sql('''
    SELECT * FROM ( SELECT
            hour,
            ROUND(AVG(COMP)*100, 1) AS phan_tram_chay,
            'CAO_DIEM' AS nhom_tai
        FROM sensor
        GROUP BY hour
        ORDER BY phan_tram_chay DESC
        LIMIT 3
    ) high_load
    UNION ALL
    SELECT * FROM ( SELECT
            hour,
            ROUND(AVG(COMP)*100, 1) AS phan_tram_chay,
            'THAP_DIEM' AS nhom_tai
        FROM sensor
        GROUP BY hour
        ORDER BY phan_tram_chay ASC
        LIMIT 3
    ) low_load
    ORDER BY phan_tram_chay DESC''')
print("EXECUTION PLAN")
q2_combined.explain(True)
print("PHÂN TÍCH 3 KHUNG GIỜ CAO ĐIỂM & 3 KHUNG GIỜ THẤP ĐIỂM")
df_q2 = q2_combined.toPandas()
try:  display(df_q2)
except NameError: print(df_q2.to_string(index=False))

EXECUTION PLAN
== Parsed Logical Plan ==
'Sort ['phan_tram_chay DESC NULLS LAST], true
+- 'Union false, false
   :- 'Project [*]
   :  +- 'SubqueryAlias high_load
   :     +- 'GlobalLimit 3
   :        +- 'LocalLimit 3
   :           +- 'Sort ['phan_tram_chay DESC NULLS LAST], true
   :              +- 'Aggregate ['hour], ['hour, 'ROUND(('AVG('COMP) * 100), 1) AS phan_tram_chay#44, CAO_DIEM AS nhom_tai#45]
   :                 +- 'UnresolvedRelation [sensor], [], false
   +- 'Project [*]
      +- 'SubqueryAlias low_load
         +- 'GlobalLimit 3
            +- 'LocalLimit 3
               +- 'Sort ['phan_tram_chay ASC NULLS FIRST], true
                  +- 'Aggregate ['hour], ['hour, 'ROUND(('AVG('COMP) * 100), 1) AS phan_tram_chay#46, THAP_DIEM AS nhom_tai#47]
                     +- 'UnresolvedRelation [sensor], [], false

== Analyzed Logical Plan ==
hour: int, phan_tram_chay: double, nhom_tai: string
Sort [phan_tram_chay#44 DESC NULLS LAST], true
+- Union false, false
   :- Projec

,hour,phan_tram_chay,nhom_tai
0,9,84.8,CAO_DIEM
1,20,84.7,CAO_DIEM
2,13,84.5,CAO_DIEM
3,2,82.2,THAP_DIEM
4,0,81.5,THAP_DIEM
5,1,81.2,THAP_DIEM
